# 0.5 Association Rule Mining

**Author:** Ahmed Hasan | **Report:** Final

Binarize 15+ neighbourhood features against medians and apply Apriori
to discover equity-relevant association rules.

In [ ]:
# Papermill parameters
min_support = 0.15
min_confidence = 0.5
min_lift = 1.0
max_len = 3
report_name = "final"


In [ ]:
from ptn_analysis.context import TransitContext
from ptn_analysis.context.config import FEED_ID_CURRENT, REPORTS_DIR
from ptn_analysis.analysis import AssociationRuleMiner
from ptn_analysis.context.reporting import save_report_figure
from ptn_analysis.analysis.visualization import plot_association_rules_network
import pandas as pd

ctx = TransitContext.from_defaults()
db = ctx.working_db
miner = AssociationRuleMiner("ywg", FEED_ID_CURRENT, db)

In [ ]:
# Build binary feature matrix from neighbourhood data
binary_df = miner.build_binary_feature_matrix()
print(f"Binary feature matrix: {binary_df.shape[0]} neighbourhoods x {binary_df.shape[1]} features")
print(f"Features: {list(binary_df.columns)}")
binary_df.head()


In [ ]:
# Run Apriori and extract association rules
rules_df = miner.mine_rules(
    min_support=min_support,
    min_confidence=min_confidence,
    min_lift=min_lift,
    max_len=max_len,
)
print(f"Discovered {len(rules_df)} rules with lift >= {min_lift}")
if not rules_df.empty:
    rules_df.sort_values("lift", ascending=False).head(15)

In [ ]:
# Plot association rules network (features as nodes, rules as edges)
import matplotlib.pyplot as plt
if not rules_df.empty:
    fig, ax = plot_association_rules_network(rules_df, top_n=20)
    save_report_figure(fig, "association_rules_network.png", report_name)
    plt.close(fig)
else:
    print("No rules discovered; adjust min_support or min_confidence.")


In [ ]:
# Export rules to parquet
export_dir = REPORTS_DIR / report_name / "tables"
export_dir.mkdir(parents=True, exist_ok=True)
if not rules_df.empty:
    rules_df.to_parquet(export_dir / "association_rules.parquet", index=False)
    print(f"Exported {len(rules_df)} rules to {export_dir / 'association_rules.parquet'}")